*(image: JHU_GL_logo.png)*

# **Project 1 - DualLens Analytics**

*(image: image.png)*

### **Background Story**

In the rapidly evolving world of finance and technology, investors are constantly seeking ways to make smarter decisions by combining traditional financial analysis with emerging technological insights. While stock market trends provide a numerical perspective on growth, an organization’s initiatives in cutting-edge fields like Artificial Intelligence (AI) reveal its future readiness and innovation potential. However, analyzing both dimensions - quantitative financial performance and qualitative AI initiatives - requires sifting through multiple, diverse data sources: stock data from platforms like Yahoo Finance, reports in PDFs, and contextual reasoning using Large Language Models (LLMs).

This is where **DualLens Analytics** comes in. By applying a dual-lens approach, the project leverages **Retrieval-Augmented Generation (RAG)** to merge **financial growth data** with **strategic insights from organizational reports**. Stock data provides evidence of stability and momentum, while AI initiative documents reveal forward-looking innovation. Together, they form a richer, more holistic picture of organizational potential.

With DualLens Analytics, investors no longer need to choose between numbers and narratives—they gain a unified, AI-driven perspective that ranks organizations by both financial strength and innovation readiness, enabling smarter, future-focused investment strategies.

### **Problem Statement**

Traditional investment analysis often focuses on financial metrics alone (e.g., stock growth, revenue, market cap), missing the qualitative dimension of how prepared a company is for the future. On the other hand, qualitative documents like strategy PDFs contain valuable insights about innovation and AI initiatives, but they are difficult to structure, query, and integrate with numeric financial data.

This leads to three core challenges:

  1. **Fragmented Data Sources:** Financial data (stock prices) and strategic insights (PDFs) exist in silos.

  2. **Limited Analytical Scope:** Manual analysis of growth trends and PDF reports is time-consuming and error-prone.

  3. **Decisional Blind Spots:** Without integrating both quantitative (growth trends) and qualitative (AI initiatives) signals, investors may miss out on high-potential organizations.

### **Solution Approach**

To address this challenge, we set out to build a **Retrieval-Augmented Generation (RAG)** powered system that blends financial trends with AI-related strategic insights, helping investors rank organizations based on growth trajectory and innovation capacity.

*(image: image.png)*

* * *

###  **# Important Points To Note** 

  1. For LLM please use **`"gpt-4o-mini"`** model.
  2. You need to look for **`"--- --- ---"`** and add your code over there, this is a placeholder.
  3. For project submission, please provide the project in **HTML format.** Refer to the **"Guidelines: Converting IPYNB to HTML Files.pdf"** for instructions.
  4. Ensure all **outputs are clearly visible,** and enclose prompts in triple quotes **`"""`** so they remain fully visible when the notebook is converted from IPYNB to HTML.



* * *

# **Setting up Installations and Imports**

In [ ]:
# @title Run this cell => Restart the session => Start executing the below cells **(DO NOT EXECUTE THIS CELL AGAIN)**

!pip install langchain==0.3.25 \
                langchain-core==0.3.65 \
                langchain-openai==0.3.24 \
                chromadb==1.3.4 \
                langchain-community==0.3.20 \
                pypdf==5.4.0

In [ ]:
import yfinance as yf              # Used for gathering stock prices
import matplotlib.pyplot as plt    # Used for Data Visualization / Plots / Graphs
import pandas as pd                # Helpful for working with tabular data like DataFrames
import os                          # Interacting with the operating system

from langchain.text_splitter import RecursiveCharacterTextSplitter      #  Helpful in splitting the PDF into smaller chunks

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader     # Loading a PDF
from langchain_community.vectorstores import Chroma    # Vector DataBase

# **1\. Organization Selection**

Selecting the below five organizations as the analysis pool.

In [ ]:
companies = ["GOOGL", "MSFT", "IBM", "NVDA", "AMZN"]

# **2\. Setting up LLM - 1 Marks**

* The `config.json` file should contain **API_KEY** and **API BASE URL** provided by OpenAI.
  * You need to insert your actual API keys and endpoint URL obtained from your Olympus account. Refer to the **OpenAI Access Token documentation** for more information on how to generate and manage your API keys.
  * This code reads the `config.json` file and extracts the API details.
    * The `API_KEY` is a **unique secret key** that authorizes your requests to OpenAI's API.
    * The `OPENAI_API_BASE` is the **API BASE URL** where the model will process your requests.

**What To Do?**

  * Use the sample `config.json` file provided.

  * Add their **OpenAI API Key** and **Base URL** to the file.

  * The `config.json` should look like this:
        
        {
              "API_KEY": "your_openai_api_key_here",
              "OPENAI_API_BASE": "https://your_openai_api_base/v1"
            }

In [ ]:
#Loading the `config.json` file
import json
import os

# Load the JSON file and extract values
file_name = "config.json"
with open(file_name, 'r') as file:
    config = json.load(file)

    os.environ['OPENAI_API_KEY'] = config.get("API_KEY")
    os.environ["OPENAI_BASE_URL"] = config.get("OPENAI_API_BASE")

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(
    model="gpt-4o-mini",                      # "gpt-4o-mini" to be used as an LLM
    temperature="0",                # Set the temprature to 0
    max_tokens="5000",                 # Set the max_tokens = 5000, so that the long response will not be clipped off
    top_p=0.95,
    frequency_penalty=1.2,
    stop_sequences=['INST']
)

# **3\. Visualization and Insight Extraction - 5 Marks**

*(image: generated-image \(2\) \(1\).png)*

Gather stock data for the selected organization from the past three years using the YFinance library, and visualize this data for enhanced analysis.

> **Your Task**

  1. Loop through each company to retrieve stock data of the last three years using the YFinance library.
  2. Plot the closing prices for each company.

In [ ]:
plt.figure(figsize=(14,7))

# Loop through each company and plot closing prices
for symbol in companies:
    ticker = yf.Ticker(symbol)
    data = ticker.history(period="3y")

    # Plot closing price
    plt.plot(data.index, data['Close'], label=symbol)

plt.title("Stock Price Trends (Last 3 Years)")
plt.xlabel("Date")
plt.ylabel("Price (USD)")
plt.legend()
plt.grid(True)
plt.savefig("Stock_Price_Trends_3Y.png")
plt.show()

#### **Financial Metrics**

  1. **Market Cap:** Total market value of a company’s outstanding shares.
  2. **P/E Ratio:** Shows how much investors are willing to pay per dollar of earnings.
  3. **Dividend Yield:** Annual dividend income as a percentage of the stock price.
  4. **Beta** : Measures a stock’s volatility relative to the overall market.
  5. **Total Revenue:** The total income a company generates from its business operations.

> **Your Task**

  1. Loop through all the companies to fetch data based on the specified financial metrics.
  2. Create a DataFrame (DF) from the collected data.
  3. Visualize and compare each financial metric across all companies.
  4. For example, visualize and compare the market capitalization for each company.



_Tip: Check`ticker.info` for the available financial metrics_

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

companies = ["GOOGL", "MSFT", "IBM", "NVDA", "AMZN"]
metrics_list = {}

# Fetching the financial metrics
for symbol in companies:                          # Loop through all the companies
    ticker = yf.Ticker(symbol)
    info = ticker.info
    metrics_list[symbol] = {                              # Define the dictionary of all the Finanical Metrics
        "Market Cap": info.get("marketCap", 0),
        "P/E Ratio": info.get("forwardPE", 0),
        "Dividend Yield": info.get("dividendYield", 0),
        "Beta": info.get("beta", 0),
        "Total Revenue": info.get("totalRevenue", 0)
    }

In [ ]:
# Convert to DataFrame
df = pd.DataFrame(metrics_list).T

# Converting large numbers to billions for readability by divinding the whole column by 1e9
df["Market Cap"] = df["Market Cap"] / 1e9
df["Total Revenue"] = df["Total Revenue"] / 1e9
df["Dividend Yield"] = df["Dividend Yield"] * 100  # Convert to percentage

df   # Printing the df

In [ ]:
# Plot each metric as a separate bar graph
metrics_to_plot = ["Market Cap", "P/E Ratio", "Dividend Yield", "Beta", "Total Revenue"]

for metric in metrics_to_plot:
    plt.figure(figsize=(10,5))
    plt.bar(df.index, df[metric], color='skyblue')
    plt.title(f"{metric} Comparison")
    plt.ylabel(metric)
    plt.xlabel("Company")
    plt.grid(axis='y')
    plt.show()

# **4\. RAG-Driven Analysis - 7 Marks**

*(image: generated-image \(1\) \(1\).png)*

Performing the RAG-Driven Analysis on the AI Initiatives of the companies

> **Your Task**

  1. Extract all PDF files from the provided ZIP file.
  2. Read the content from each PDF file.
  3. Split the content into manageable chunks.
  4. Store the chunks in a vector database using embedding functions.
  5. Implement a query mechanism on the vector database to retrieve results based on user queries regarding AI initiatives.
  6. Evaluate the LLM generated response using LLM-as-Judge

### **A. Loading Company AI Initiative Documents (PDFs) - 1 mark**

In [ ]:
# Unzipping the AI Initiatives Documents
import zipfile
with zipfile.ZipFile("Companies-AI-Initiatives.zip", 'r') as zip_ref:
  zip_ref.extractall("/content/")         # Storing all the unzipped contents in this location

In [ ]:
# Path of all AI Initiative Documents
ai_initiative_pdf_paths = [f"/content/Companies-AI-Initiatives/{file}" for file in os.listdir("/content/Companies-AI-Initiatives")]
ai_initiative_pdf_paths

In [ ]:
from langchain_community.document_loaders import PyPDFDirectoryLoader
loader = PyPDFDirectoryLoader(path = "/content/Companies-AI-Initiatives/")          # Creating an PDF loader object

In [ ]:
# Defining the text splitter
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1000,
    chunk_overlap=200
)

In [ ]:
# Splitting the chunks using the text splitter
ai_initiative_chunks = loader.load_and_split(text_splitter)

In [ ]:
# Total length of all the chunks
len(ai_initiative_chunks)

### **B. Vectorizing AI Initiative Documents with ChromaDB - 1 mark**

In [ ]:
# Defining the 'text-embedding-ada-002' as the embedding model
from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-ada-002")

In [ ]:
#  Creating a Vectorstore, storing all the above created chunks using an embedding model
vectorstore = Chroma.from_documents(
    ai_initiative_chunks,
    embedding_model,
    collection_name="AI_Initiatives"
)

# Ignore if it gives an error or warning

In [ ]:
# Creating an retriever object which can fetch ten similar results from the vectorstore
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 10}
)

### **C. Retrieving relevant Documents - 3 marks**

In [ ]:
user_message = "Give me the best project that `IBM` company is working upon"

In [ ]:
# Building the context for the query using the retrieved chunks
relevant_document_chunks = retriever.get_relevant_documents(user_message)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [ ]:
len(relevant_document_chunks)

In [ ]:
# Write a system message for an LLM to help craft a response from the provided context
qna_system_message = """
You are a helpful and informative Q&A assistant. Your task is to answer questions based on the provided context only.
If the answer to a question is not found in the context, please state that you don't have enough information to answer the question.
"""

In [ ]:
# Write an user message template which can be used to attach the context and the questions
qna_user_message_template = """
###Context
Here are some documents that are relevant to the question mentioned below.
{context}

###Question
{question}
"""

In [ ]:
# Format the prompt
formatted_prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context="--- --- ---", question="--- --- ---")}
                [/INST]"""

In [ ]:
# Make the LLM call
resp = llm.invoke(formatted_prompt)
resp.content

In [ ]:
def RAG(user_message):
    """
    Args:
    user_message: Takes a user input for which the response should be retrieved from the vectorDB.
    Returns:
    relevant context as per user query.
    """
    relevant_document_chunks = retriever.get_relevant_documents(user_message)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine qna_system_message and qna_user_message_template to create the prompt
    user_message_formatted = qna_user_message_template.format(context=context_for_query, question=user_message)
    prompt = f"""[INST]{qna_system_message}
                {'user'}: {user_message_formatted}
                [/INST]"""

    # Quering the LLM
    try:
        response = llm.invoke(prompt)

    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response.content

In [ ]:
# Test Cases
print(RAG("How is the area in which GOOGL is working different from the area in which MSFT is working?"))

In [ ]:
print(RAG("What are the three projects on which MSFT is working upon?"))

In [ ]:
# Writing the system message and the evaluation metrics for checking the relevance
relevance_rater_system_message = """
You are an expert at evaluating the relevance of a generated answer to a given question. Your task is to analyze the provided answer and determine how well it addresses the question, based on the given context.

YOU MUST FOLLOW THESE RULES:
1.  Read the question and the context carefully.
2.  Read the answer and identify the main points it conveys.
3.  Determine if the main points of the answer directly address the question.
4.  Ignore any factual correctness; focus solely on whether the answer is pertinent to the question.
5.  If the answer contains information that is not directly related to the question, even if it's from the context, it should be considered less relevant.

Based on your analysis, provide a score for relevance on a scale of 1 to 5, where 1 means the answer is completely irrelevant to the question, and 5 means the answer is perfectly relevant and directly answers the question.

Also, provide a brief explanation for your score, specifically highlighting why you consider the answer to be relevant or irrelevant to the question.

"""

In [ ]:
# Combining relevance_rater_system_message + llm_prompt + answer for evaluation
# Assume evaluation_test_question and answer are already defined globally for this evaluation step
# Example definition for evaluation_test_question and answer:
# evaluation_test_question = "What is the timeline of each project in NVDA?"
# answer = "The timeline for each project in NVDA is not explicitly provided in the context."

# Retrieve relevant documents to create context_for_query for evaluation
relevant_document_chunks_eval = retriever.get_relevant_documents(evaluation_test_question)
context_list_eval = [d.page_content for d in relevant_document_chunks_eval]
context_for_query = ". ".join(context_list_eval)

relevance_prompt = f"""[INST]{relevance_rater_system_message}
            {'user'}: {evaluation_user_message_template.format(context=context_for_query, question=evaluation_test_question, answer=answer)}
            [/INST]"""

In [ ]:
print(RAG("What are the risks associated with projects within GOOG?"))

### **D. Evaluation of the RAG - 2 marks**

In [ ]:
# Writing a question for performing evaluations on the RAG
evaluation_test_question = "What are the three projects on which MSFT is working upon?"

In [ ]:
# Building the context for the evaluation test question using the retrieved chunks
relevant_document_chunks = retriever.get_relevant_documents(evaluation_test_question)
context_list = [d.page_content for d in relevant_document_chunks]
context_for_query = ". ".join(context_list)

In [ ]:
# Default RAG Answer
answer = RAG(evaluation_test_question)
print(answer)

In [ ]:
# Defining user messsage template for evaluation
evaluation_user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

##### **1\. Groundedness**

In [ ]:
# Writing the system message and the evaluation metrics for checking the groundedness
groundedness_rater_system_message = """
You are an expert at evaluating the groundedness of a generated answer given a question and a context. Your task is to analyze the provided answer and determine if all the information presented in the answer is directly supported by the provided context.

YouMust follow these rules:
1.  Read the context and the question carefully.
2.  Read the answer and identify each factual statement within it.
3.  For each factual statement, determine if it can be directly inferred from the provided context.
4.  If a statement is not supported by the context, it is considered a hallucination.
5.  If any part of the answer contains information not present in the context, even if it is factually correct, you must identify it as ungrounded.
6.  The answer should only contain information found in the context.

Based on your analysis, provide a score for groundedness on a scale of 1 to 5, where 1 means the answer is completely ungrounded or contains significant hallucinations, and 5 means the answer is perfectly grounded and all information comes directly from the context.

Also, provide a brief explanation for your score, specifically highlighting any parts of the answer that are not grounded in the context.

"""

In [ ]:
# Combining groundedness_rater_system_message + llm_prompt + answer for evaluation
groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
            {'user'}: {evaluation_user_message_template.format(context=context_for_query, question=evaluation_test_question, answer=answer)}
            [/INST]"""

In [ ]:
# Defining a new LLM object
groundness_checker = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=500,
    top_p=0.95,
    frequency_penalty=1.2,
    stop_sequences=['INST']
)

# Using the LLM-as-Judge for evaluating Groundedness
groundness_response = groundness_checker.invoke(groundedness_prompt)
print(groundness_response.content)

##### **2\. Relevance**

In [ ]:
# Writing the system message and the evaluation metrics for checking the relevance
relevance_rater_system_message = """
You are an expert evaluator assessing the relevance of an AI-generated answer to a specific user question.

Your task is to determine how well the answer addresses the user's question, strictly based on the provided context.
Follow these rules:
1. Ignore factual correctness (groundedness is evaluated separately). Focus solely on whether the answer directly and appropriately responds to the prompt.
2. If the answer includes extra information not requested by the prompt, penalize the score slightly.
3. Score the relevance on a scale of 1 to 5, where 1 means completely irrelevant to the question, and 5 means perfectly aligned and directly answers the question.

Provide a brief explanation for your score, detailing why the answer hit or missed the mark.
"""

In [ ]:
# Combining relevance_rater_system_message + llm_prompt + answer for evaluation
relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
            {'user'}: {evaluation_user_message_template.format(context=context_for_query, question=evaluation_test_question, answer=answer)}
            [/INST]"""

In [ ]:
# Defining a new LLM object
relevance_checker = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
    max_tokens=500,
    top_p=0.95,
    frequency_penalty=1.2,
    stop_sequences=['INST']
)

# Using the LLM-as-Judge for evaluating Relevance
relevance_response = relevance_checker.invoke(relevance_prompt)
print(relevance_response.content)

# **5\. Scoring and Ranking - 3 Marks**

*(image: image.png)*

Prompting an LLM to score each company by integrating Quantitative data (stock trend, growth metrics) and Qualitative evidence (PDF insights)

> **Your Task**

  1. Write a system message and a user message that outlines the required data for the prompt.
  2. Prompt the LLM to rank and recommend companies for investment based on the provided PDF and stock data to achieve better returns.

In [ ]:
# Fetching all the links of the documents
len(vectorstore.get()['documents'])

In [ ]:
# Write a user message for instructing the LLM for scoring and ranking the companies

system_message = """
You are an expert financial analyst and AI strategist. Your objective is to evaluate and rank five tech companies (GOOGL, MSFT, IBM, NVDA, AMZN) for potential investment.
Analyze the companies using a "Dual-Lens" approach: synthesize both their quantitative financial health and their qualitative future readiness (AI initiatives).
"""

user_message = f"""
Please rank the companies from 1 (Most Favorable Investment) to 5 (Least Favorable Investment).
Provide a clear, step-by-step justification for your ranking. For each company, highlight exactly how both their numerical financial strength and their AI narratives influenced your final decision based on the data below.

---
### 1. Financial Data
{df.to_string()}

---
### 2. AI Initiatives
{vectorstore.get()['documents']}
"""

In [ ]:
# Formatting the prompt
formatted_prompt = f"""[INST]{system_message}\n
                {'user'}: {user_message}
                [/INST]"""

In [ ]:
# Calling the LLM
recommendation = llm.invoke(formatted_prompt)
recommendation.content

In [ ]:
print(recommendation.content)

# **6\. Summary and Future Scope - 4 Marks**

Based on the project, learners are expected to share their observations, key learnings, and insights related to the business use case, including any challenges they encountered. Additionally, they should recommend improvements to the project and suggest further steps for enhancement.

**A. Summary / Your Observations about this Project - 2 Marks**

1. Holistic "Dual-Lens" Effectiveness: By integrating both quantitative financial data and qualitative AI initiatives, the system effectively avoids "decisional blind spots." For instance, it successfully identified Microsoft (MSFT) as the top overall investment by correlating its attractive financial valuation (P/E 21.07) with its aggressive AI rollout strategy (Azure AI Foundry, Copilot).

  2. LLM Synthesis & Reasoning Capabilities: The gpt-4o-mini model demonstrated an excellent capacity to act as a financial analyst. It was able to ingest structured tabular data (the Pandas DataFrame) alongside unstructured RAG chunks and output a deeply reasoned 1-to-5 ranking, penalizing companies like IBM for struggling to maintain a competitive dynamic edge despite having a high dividend yield.

  3. The RAG Context Bottleneck: A key technical challenge encountered was ensuring the vector database retrieved the most pertinent strategic information. The final investment ranking is highly sensitive to the quality of the chunks retrieved by ChromaDB; if key AI milestones are missed during retrieval, a company could be unfairly downranked in the final synthesis.

**B. Future scope of this Project - 2 Marks**

1. Real-Time Data Integration: The project currently relies on static PDF reports and historical stock trends. A crucial improvement would be integrating real-time news APIs, live SEC filings (10-K, 10-Q), and quarterly earnings call transcripts to capture immediate market sentiment and sudden strategic shifts.

  2. Advanced Retrieval Techniques (Hybrid Search): To improve context accuracy for the LLM, the system could implement "Hybrid Search"—combining semantic vector embeddings with keyword-based BM25 search. This ensures highly specific technical or financial terms are always retrieved from the documents.

  3. Automated Portfolio Weighting: Instead of a simple 1-to-5 ranking, the final LLM prompt could be enhanced to output suggested portfolio allocation percentages (e.g., 40% MSFT, 25% GOOGL, 15% NVDA) customized to different investor risk profiles (conservative vs. aggressive).